# 00 — The Python Data Model

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('advanced/00', {1: 'ee93322b20999c11', 2: 'dee464e0f6eaca8a', 3: 'a57fd54d30be6d0d', 4: 'c31cb3f9cc863e64', 5: '9e464be3da7d309b', 6: '3a5890ae486451ca', 7: 'a9f58adfa4c9cb38', 8: '5fa6d4f56cd74a1d', 9: '8586ac092841585d', 10: '10f5d888679c1441', 11: '2ef72e12743dbda5', 12: '3e6e1790d026dc84'}, practice='advanced_01_internals_typing_architecture')

## 1. Collection protocols

- Python's syntax calls special methods: `len(x)` → `x.__len__()`, `for` → `__iter__`, `in` → `__contains__`, `x[i]` → `__getitem__`.
- Copy mutable input in `__init__` (`list(items)`) so the caller can't change your object behind your back.
- `__repr__` is for developers: unambiguous, ideally looks like the constructor call, and short for big objects.

In [ ]:
class Bag:
    def __init__(self, items):
        self._items = list(items)
    def __len__(self):
        return len(self._items)
    def __iter__(self):
        return iter(self._items)
    def __contains__(self, item):
        return item in self._items
    def __repr__(self):
        return f"Bag({self._items!r})"

source = [1, 2]
b = Bag(source)
source.append(99)
print(len(b), 2 in b, list(b), b)

**Exercise 1.** Which method does `len(obj)` call? Write its name.

In [ ]:
answer = ___
check(1, answer)

**Exercise 2.** Make `in` work by delegating to the stored list.

In [ ]:
class Bag:
    def __init__(self, items):
        self._items = list(items)
    def __contains__(self, item):
        return item in ___

b = Bag(["a", "b"])
check(2, ("a" in b, "z" in b))

**Exercise 3.** The bag copies its input. After `source.append(3)`, what is `len(bag)`?

In [ ]:
class Bag:
    def __init__(self, items): self._items = list(items)
    def __len__(self): return len(self._items)

source = [1, 2]
bag = Bag(source)
source.append(3)
predicted = ___
check(3, predicted)

**Exercise 4.** Show at most 3 items in the repr: `Bag([1, 2, 3, ...])`.

In [ ]:
items = [1, 2, 3, 4, 5]
shown = ", ".join(repr(i) for i in items[:___])
more = ", ..." if len(items) > 3 else ""
check(4, f"Bag([{shown}{more}])")

## 2. Equality, hashing, ordering

- If you define `__eq__`, define `__hash__` from the same fields: equal objects must have equal hashes, or they misbehave in sets and dicts.
- Compare via a key tuple: `def _key(self): return (self.priority, self.sequence)`.
- For a type you don't support, `return NotImplemented` (don't raise, don't return `False`). Python then tries the other side or raises `TypeError`.

In [ ]:
class Ticket:
    def __init__(self, priority, sequence):
        self.priority, self.sequence = priority, sequence
    def _key(self):
        return (self.priority, self.sequence)
    def __eq__(self, other):
        if not isinstance(other, Ticket):
            return NotImplemented
        return self._key() == other._key()
    def __lt__(self, other):
        if not isinstance(other, Ticket):
            return NotImplemented
        return self._key() < other._key()
    def __hash__(self):
        return hash(self._key())

ts = [Ticket(2, 1), Ticket(1, 5), Ticket(1, 2)]
print([t._key() for t in sorted(ts)], len({Ticket(1, 1), Ticket(1, 1)}))
print(Ticket(1, 1) == "x")

**Exercise 5.** Sort tickets: lower priority first, then by sequence.

In [ ]:
pairs = [(2, 1), (1, 5), (1, 2)]
ordered = sorted(pairs, key=lambda p: ___)
check(5, ordered)

**Exercise 6.** What should `__eq__` return for an unsupported type? Write it as text.

In [ ]:
answer = ___
check(6, answer)

**Exercise 7.** With `__eq__` returning `NotImplemented` for strings, what is `Ticket(1, 1) == "x"`?

In [ ]:
predicted = ___
check(7, predicted)

**Exercise 8.** Two equal tickets go into a set. How many items does it hold?

In [ ]:
class T:
    def __init__(self, k): self.k = k
    def __eq__(self, o): return isinstance(o, T) and self.k == o.k
    def __hash__(self): return hash(self.k)

predicted = ___   # len({T(1), T(1), T(2)})
check(8, predicted)

## 3. Arithmetic operators

- `a + b` calls `a.__add__(b)`; if that returns `NotImplemented`, Python tries `b.__radd__(a)`.
- `3 * v` needs `__rmul__`; for a symmetric operation, `__rmul__ = __mul__`.
- `abs(v)` calls `__abs__`; `math.hypot(x, y)` is the length of a 2-D vector. `__slots__` saves memory and blocks new attributes.

In [ ]:
import math
class Vector:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __add__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x + other.x, self.y + other.y)
    def __mul__(self, factor):
        if not isinstance(factor, (int, float)):
            return NotImplemented
        return Vector(self.x * factor, self.y * factor)
    __rmul__ = __mul__
    def __abs__(self):
        return math.hypot(self.x, self.y)
    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

print(Vector(1, 2) + Vector(3, 4), 2 * Vector(1, 1), abs(Vector(3, 4)))

**Exercise 9.** Length of the vector (3, 4).

In [ ]:
import math
length = math.hypot(3, ___)
check(9, length)

**Exercise 10.** Which method does `2 * v` try on the vector after `int.__mul__` gives up? Write its name.

In [ ]:
answer = ___
check(10, answer)

**Exercise 11.** Finish `__add__` for 2-D vectors stored as tuples.

In [ ]:
def add(a, b):
    return (a[0] + b[0], ___)

check(11, add((1, 2), (3, 4)))

**Exercise 12.** A class with `__slots__ = ("x", "y")`: can you set `v.z = 1`? Write `True` or `False`.

In [ ]:
class V:
    __slots__ = ("x", "y")

v = V()
try:
    v.z = 1
    allowed = True
except AttributeError:
    allowed = False
predicted = ___
check(12, predicted)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### 1. `ReadOnlyBag`

A read-only collection with `len`, iteration, `in` and a short repr.

In [ ]:
class ReadOnlyBag:
    """A read-only collection supporting len(), iteration, `in` and a safe repr.

    It copies its input so later changes to the caller's list do not leak in.
    repr shows at most 3 items: ReadOnlyBag([1, 2, 3, ...]) for longer bags.
    """

    def __init__(self, items: Iterable):
        self._items = ___

    def __len__(self) -> int:
        return ___

    def __iter__(self) -> Iterator:
        return ___

    def __contains__(self, item) -> bool:
        return ___

    def __repr__(self) -> str:
        shown = ", ".join(repr(i) for i in self._items[:3])
        more = ", ..." if len(self._items) > 3 else ""
        return ___

In [ ]:
check('ReadOnlyBag')

### 2. `Ticket`

Order by priority then sequence; hashable; `NotImplemented` for other types.

In [ ]:
class Ticket:
    """Ordered by priority (lower first), then by sequence number.

    __eq__ and __lt__ return NotImplemented for non-Ticket operands, and
    Tickets are hashable on (priority, sequence).
    """

    def __init__(self, priority: int, sequence: int, title: str = ""):
        self.priority, self.sequence, self.title = ___

    def _key(self) -> tuple[int, int]:
        return ___

    def __eq__(self, other):
        if ___:
            return NotImplemented
        return self._key() == other._key()

    def __lt__(self, other):
        if ___:
            return NotImplemented
        return self._key() < other._key()

    def __hash__(self) -> int:
        return ___

In [ ]:
check('Ticket')

### Independent build — `Vector`

Implement Vector(x, y) with coordinates x and y and no per-instance __dict__. Support vector addition and subtraction, scalar multiplication from either side, Euclidean magnitude, coordinate equality and matching hashes. Unsupported arithmetic operands return NotImplemented from the special method. repr(Vector(3, 4)) is 'Vector(3, 4)'.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
class Vector:
    pass  # Implement the public interface in the ticket.

In [ ]:
check('Vector')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** If you define __eq__, you should also think about…

   - **a)** __len__
   - **b)** __iter__
   - **c)** __hash__

**2.** What should __lt__ return for an unsupported type?

   - **a)** raise TypeError directly
   - **b)** NotImplemented
   - **c)** False

**3.** `a is b` checks…

   - **a)** The same object
   - **b)** The same type
   - **c)** Equal values

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/00', {1: '21b43ee94f7f', 2: 'c5dcf34c1dfb', 3: 'b43d4be3104b'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [01 — Descriptors, Properties, and Metaclasses](01_descriptors_and_metaclasses.ipynb)